# Transformer input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30, 39, or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [1]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_transformer_imlive.py and updated transformer files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 30              # 30, 39, or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['axial']  # Add 'dq' to run the global-attention model too
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 16
D_MODEL = 96
DEPTH = 4
NUM_HEADS = 6
DROPOUT = 0.1
WARMUP_EPOCHS = 5
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_transformer_imlive.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case30_dq


In [2]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_transformer_imlive.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE,
    '--d-model', str(D_MODEL), '--depth', str(DEPTH), '--num-heads', str(NUM_HEADS),
    '--dropout', str(DROPOUT), '--warmup-epochs', str(WARMUP_EPOCHS)]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case30_dq --models axial --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 200 --batch-size 16 --loss-alpha 0.5 --device cuda --d-model 96 --depth 4 --num-heads 6 --dropout 0.1 --warmup-epochs 5
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6880, 3, 30, 30), y=(6880, 1, 30, 30)) 
 Dataset(dq, X=(1474, 3, 30, 30), y=(1474, 1, 30, 30))

axial ddc seed=42: 901,153 parameters; AMP=True
epoch=  1 loss=0.16131 val_R2=0.8739049300037717 val_RMSE=1.515813 (8.3s)
epoch=  2 loss=0.14317 val_R2=0.8752097443968709 val_RMSE=1.507949 (8.1s)
epoch=  3 loss=0.14250 val_R2=0.8757819432946216 val_RMSE=1.504488 (8.1s)
epoch=  4 loss=0.14

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/train_transformer_imlive.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case30', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case30_dq', '--models', 'axial', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '200', '--batch-size', '16', '--loss-alpha', '0.5', '--device', 'cuda', '--d-model', '96', '--depth', '4', '--num-heads', '6', '--dropout', '0.1', '--warmup-epochs', '5'], returncode=0)

In [8]:
from pathlib import Path
import subprocess
import sys
import json
import csv
OUT_DIR=Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Axial_transformer/results/case30_dq')
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
axial ddc test: {'r2': {'mean': 0.8785528664183735, 'std_across_seeds': 0.0003965121848340823}, 'rmse': {'mean': 1.5018679083222475, 'std_across_seeds': 0.00245152158340369}, 'mae': {'mean': 0.6168791095165425, 'std_across_seeds': 0.008602935375011046}}
axial ddc_yb test: {'r2': {'mean': 0.8822207627707023, 'std_across_seeds': 0.0006244089675553627}, 'rmse': {'mean': 1.4790124879610256, 'std_across_seeds': 0.003919040141769714}, 'mae': {'mean': 0.6042938618723875, 'std_across_seeds': 0.002840983849117637}}
axial ddc_imag test: {'r2': {'mean': 0.8802618525128434, 'std_across_seeds': 0.00042589346769386687}, 'rmse': {'mean': 1.4912631900389197, 'std_across_seeds': 0.0026527431451408754}, 'mae': {'mean': 0.6096701584789419, 'std_across_seeds': 0.0036922248839028754}}
axial ddc_real test: {'r2': {'mean': 0.8806207825898428, 'std_across_seeds': 0.00014671338169118848}, 'rmse': {'mean': 1.4890277717323652, 'std_across_seeds': 0.0009148908765893982}, 'mae': 

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. Axial attention is the default. Global attention over B*B cells has much higher memory use; reduce batch size if selecting dq. See README.md for inference and recovery.
